# Turbulent Cylinder Flow: High Reynolds Number Simulation

This notebook demonstrates the **turbulent cylinder flow model** with:
- **Parameterized cylinder position** (cx, cy)
- **Smagorinsky LES** turbulence modeling
- **Interactive visualization** with ipywidgets

## Requirements

```bash
conda install -c conda-forge fenics-dolfinx mpich pyvista ipywidgets
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.colors import Normalize

import sys
sys.path.insert(0, '..')

# Check for widgets
try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, IntSlider, FloatSlider, Dropdown, Checkbox, Play, HBox, VBox
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Install with: pip install ipywidgets")

# Check for FEniCSx
try:
    from pdeforge import get_model, list_models
    if 'cylinder_flow_2d_turbulent' in list_models():
        HAS_MODEL = True
        print("Turbulent cylinder flow model available.")
    else:
        HAS_MODEL = False
        print("Turbulent model not found. Check FEniCSx installation.")
except Exception as e:
    HAS_MODEL = False
    print(f"Import error: {e}")

## 1. Understanding Reynolds Number and Turbulence

The flow regime around a cylinder depends on the **Reynolds number**:

$$Re = \frac{U \cdot D}{\nu}$$

| Re Range | Flow Regime |
|----------|-------------|
| Re < 5 | Creeping flow (no separation) |
| 5-40 | Steady separated flow |
| 40-200 | Laminar vortex shedding |
| 200-300,000 | **Turbulent wake** |
| >300,000 | Fully turbulent |

In [ ]:
def compute_reynolds(U, D, nu):
    """Compute Reynolds number."""
    return U * D / nu

# Default parameters
U = 1.0      # inlet velocity [m/s]
D = 0.1      # cylinder diameter [m]
nu = 0.0001  # kinematic viscosity [m²/s]

Re = compute_reynolds(U, D, nu)
print(f"Default Reynolds number: Re = {Re:.0f}")
print(f"This is in the TURBULENT WAKE regime.")

## 2. Model Configuration

The turbulent model uses **Smagorinsky LES** (Large Eddy Simulation) to capture turbulent fluctuations:

$$\nu_t = (C_s \Delta)^2 |S|$$

where $C_s \approx 0.1$ is the Smagorinsky constant and $\Delta$ is the mesh size.

In [ ]:
if HAS_MODEL:
    # Create model with moderate resolution for demo
    Model = get_model('cylinder_flow_2d_turbulent')
    
    # Configuration for Re ~ 500 (moderate turbulence)
    # Using reduced time steps and shorter simulation for demo speed
    model = Model(
        resolution={'x': 110, 'y': 41},
        inlet_velocity=1.0,
        viscosity=0.0002,  # Re = 1.0 * 0.1 / 0.0002 = 500
        cylinder_radius=0.05,
        cx_range=(0.15, 0.5),
        cy_range=(0.15, 0.26),
        use_les=True,
        smagorinsky_constant=0.1,
        time_end=2.0,
        n_time_steps=11,
        _mesh_resolution=0.02,
    )
    
    print(model.describe())

## 3. Generate Turbulent Flow Trajectory

This will take a few minutes due to time-stepping through the turbulent regime.

In [ ]:
if HAS_MODEL:
    print("Generating turbulent flow trajectory...")
    print("(This may take several minutes)")
    
    # Generate with default cylinder position
    trajectory = model.solve(inlet_scale=1.0, cx=0.2, cy=0.2)
    
    x = model.grids['x']
    y = model.grids['y']
    t = np.linspace(0, model.time_end, model.n_time_steps)
    
    print(f"\nTrajectory shape: {trajectory.shape}")
    print(f"  (n_time={trajectory.shape[0]}, nx={trajectory.shape[1]}, ny={trajectory.shape[2]}, channels={trajectory.shape[3]})")
    print(f"\nReynolds number: {model.Re:.0f}")

## 4. Visualization Utilities

In [ ]:
# Geometry
CYLINDER_CENTER = (0.2, 0.2)
CYLINDER_RADIUS = 0.05
CHANNEL_LENGTH = 2.2
CHANNEL_HEIGHT = 0.41

def add_cylinder(ax, center=CYLINDER_CENTER, radius=CYLINDER_RADIUS):
    """Add cylinder patch to axes."""
    circle = Circle(center, radius, color='gray', ec='black', lw=2, zorder=10)
    ax.add_patch(circle)

def compute_vorticity(u, v, x, y):
    """Compute vorticity from velocity field."""
    dx = x[1] - x[0]
    dy = y[1] - y[0]
    dvdx = np.gradient(v, dx, axis=1)
    dudy = np.gradient(u, dy, axis=0)
    return dvdx - dudy

def plot_frame(ax, u, v, p, x, y, t_val, cx, cy, field='vorticity', show_colorbar=True):
    """Plot a single frame of the flow."""
    ax.clear()
    
    if field == 'velocity':
        vmag = np.sqrt(u**2 + v**2)
        vmax = vmag.max()
        im = ax.contourf(x, y, vmag, levels=30, cmap='viridis', vmin=0, vmax=vmax)
        label = '|u| (m/s)'
        
    elif field == 'vorticity':
        omega = compute_vorticity(u, v, x, y)
        vmax = np.abs(omega).max()
        im = ax.contourf(x, y, omega, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        label = 'ω (1/s)'
        
    elif field == 'pressure':
        vmax = np.abs(p).max()
        if vmax < 1e-10: vmax = 1.0
        im = ax.contourf(x, y, p, levels=30, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
        label = 'p (Pa)'
        
    elif field == 'turbulent_ke':
        # Approximate turbulent kinetic energy from velocity fluctuations
        vmag = np.sqrt(u**2 + v**2)
        tke = 0.5 * vmag**2
        im = ax.contourf(x, y, tke, levels=30, cmap='hot')
        label = 'TKE (m²/s²)'
    
    add_cylinder(ax, center=(cx, cy), radius=CYLINDER_RADIUS)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_title(f't = {t_val:.3f} s')
    ax.set_aspect('equal')
    ax.set_xlim(0, CHANNEL_LENGTH)
    ax.set_ylim(0, CHANNEL_HEIGHT)
    
    return im, label

## 5. Time Snapshots of Turbulent Flow

In [ ]:
if HAS_MODEL:
    n_snapshots = 6
    snapshot_indices = np.linspace(0, len(t) - 1, n_snapshots, dtype=int)

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()

    for i, t_idx in enumerate(snapshot_indices):
        u = trajectory[t_idx, :, :, 0].T
        v = trajectory[t_idx, :, :, 1].T
        p = trajectory[t_idx, :, :, 2].T
        
        im, label = plot_frame(axes[i], u, v, p, x, y, t[t_idx], 0.2, 0.2, field='vorticity')
        plt.colorbar(im, ax=axes[i], label=label, shrink=0.8)

    fig.suptitle(f'Turbulent Cylinder Flow: Vorticity (Re = {model.Re:.0f})', fontsize=14)
    plt.tight_layout()
    plt.show()

## 6. Interactive Time Slider

Use the slider to explore the turbulent flow dynamics.

In [ ]:
if HAS_MODEL and HAS_WIDGETS:
    def explore_turbulent_flow(time_idx, field='vorticity'):
        """Interactive exploration of turbulent flow."""
        u = trajectory[time_idx, :, :, 0].T
        v = trajectory[time_idx, :, :, 1].T
        p = trajectory[time_idx, :, :, 2].T
        t_val = t[time_idx]
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        
        # Left: selected field
        im, label = plot_frame(axes[0], u, v, p, x, y, t_val, 0.2, 0.2, field=field)
        plt.colorbar(im, ax=axes[0], label=label, shrink=0.8)
        
        # Right: velocity magnitude with streamlines
        vmag = np.sqrt(u**2 + v**2)
        im2 = axes[1].contourf(x, y, vmag, levels=30, cmap='viridis')
        axes[1].streamplot(x, y, u, v, color='white', density=1.2, linewidth=0.5)
        add_cylinder(axes[1], center=(0.2, 0.2))
        axes[1].set_xlabel('x (m)')
        axes[1].set_ylabel('y (m)')
        axes[1].set_title('Velocity Magnitude with Streamlines')
        axes[1].set_aspect('equal')
        axes[1].set_xlim(0, CHANNEL_LENGTH)
        axes[1].set_ylim(0, CHANNEL_HEIGHT)
        plt.colorbar(im2, ax=axes[1], label='|u| (m/s)', shrink=0.8)
        
        plt.tight_layout()
        plt.show()
        
        # Statistics
        omega = compute_vorticity(u, v, x, y)
        print(f"Time: {t_val:.3f} / {model.time_end:.1f} s")
        print(f"Max velocity: {vmag.max():.4f} m/s")
        print(f"Max vorticity: {np.abs(omega).max():.2f} 1/s")
        print(f"Reynolds number: {model.Re:.0f}")
    
    time_slider = IntSlider(
        min=0, max=len(t) - 1, step=1, value=len(t) // 2,
        description='Time step:', continuous_update=False
    )
    field_dropdown = Dropdown(
        options=['vorticity', 'velocity', 'pressure', 'turbulent_ke'],
        value='vorticity', description='Field:'
    )
    
    interactive_flow = interactive(
        explore_turbulent_flow,
        time_idx=time_slider,
        field=field_dropdown,
    )
    display(interactive_flow)
else:
    print("Interactive widgets not available.")

## 7. Animation with Play Button

In [ ]:
if HAS_MODEL and HAS_WIDGETS:
    play = Play(
        value=0,
        min=0,
        max=len(t) - 1,
        step=1,
        interval=100,
        description="Play",
    )
    
    anim_slider = IntSlider(
        min=0, max=len(t) - 1, step=1, value=0,
        description='Frame:',
        continuous_update=True,
        readout=False,
    )
    
    widgets.jslink((play, 'value'), (anim_slider, 'value'))
    
    field_select = Dropdown(
        options=['vorticity', 'velocity', 'pressure'],
        value='vorticity', description='Field:'
    )
    
    output = widgets.Output()
    
    def update_animation(change):
        with output:
            clear_output(wait=True)
            
            t_idx = anim_slider.value
            field = field_select.value
            
            u = trajectory[t_idx, :, :, 0].T
            v = trajectory[t_idx, :, :, 1].T
            p = trajectory[t_idx, :, :, 2].T
            t_val = t[t_idx]
            
            fig, ax = plt.subplots(1, 1, figsize=(14, 5))
            im, label = plot_frame(ax, u, v, p, x, y, t_val, 0.2, 0.2, field=field)
            plt.colorbar(im, ax=ax, label=label, shrink=0.8)
            
            # Progress and Re
            progress = t_val / model.time_end
            ax.text(0.02, 0.95, f'Re = {model.Re:.0f} | Progress: {progress*100:.0f}%', 
                   transform=ax.transAxes, fontsize=10, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
            
            plt.tight_layout()
            plt.show()
    
    anim_slider.observe(update_animation, names='value')
    field_select.observe(update_animation, names='value')
    
    controls = HBox([play, anim_slider, field_select])
    display(VBox([controls, output]))
    
    update_animation(None)
else:
    print("Animation requires ipywidgets.")

## 8. Effect of Cylinder Position

The model supports **parameterized cylinder position**. Let's compare flows with different cylinder positions.

In [ ]:
if HAS_MODEL:
    # Note: This cell takes significant time to run
    # Uncomment to generate flows with different cylinder positions
    
    # positions = [
    #     (0.2, 0.2),   # Default position
    #     (0.3, 0.2),   # Shifted downstream
    #     (0.2, 0.25),  # Shifted toward top
    # ]
    # 
    # trajectories = {}
    # for cx, cy in positions:
    #     print(f"Generating flow for cylinder at ({cx}, {cy})...")
    #     traj = model.solve(inlet_scale=1.0, cx=cx, cy=cy)
    #     trajectories[(cx, cy)] = traj
    
    print("Cylinder position comparison: uncomment the code above to run.")
    print("This generates multiple trajectories and takes several minutes.")

## 9. Interactive Cylinder Position Explorer

Explore different cylinder positions interactively (requires pre-computed trajectories).

In [ ]:
if HAS_MODEL and HAS_WIDGETS:
    # Create sliders for cylinder position
    # Note: Changing position requires re-running simulation
    
    cx_slider = FloatSlider(
        value=0.2, min=0.15, max=0.5, step=0.05,
        description='Cylinder X:', continuous_update=False
    )
    cy_slider = FloatSlider(
        value=0.2, min=0.15, max=0.26, step=0.02,
        description='Cylinder Y:', continuous_update=False
    )
    
    position_output = widgets.Output()
    
    def show_cylinder_position(cx, cy):
        """Show domain with cylinder at specified position."""
        with position_output:
            clear_output(wait=True)
            
            fig, ax = plt.subplots(figsize=(12, 4))
            
            # Draw channel
            ax.set_xlim(0, CHANNEL_LENGTH)
            ax.set_ylim(0, CHANNEL_HEIGHT)
            ax.set_aspect('equal')
            
            # Draw cylinder at new position
            circle = Circle((cx, cy), CYLINDER_RADIUS, color='steelblue', ec='black', lw=2)
            ax.add_patch(circle)
            
            # Mark inlet/outlet
            ax.annotate('Inlet', xy=(0, CHANNEL_HEIGHT/2), fontsize=10)
            ax.annotate('Outlet', xy=(CHANNEL_LENGTH-0.2, CHANNEL_HEIGHT/2), fontsize=10)
            
            ax.set_xlabel('x (m)')
            ax.set_ylabel('y (m)')
            ax.set_title(f'Cylinder Position: ({cx:.2f}, {cy:.2f})')
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            # Compute expected Re and wake characteristics
            print(f"Cylinder center: ({cx:.2f}, {cy:.2f}) m")
            print(f"Distance from inlet: {cx:.2f} m")
            print(f"Distance from centerline: {abs(cy - CHANNEL_HEIGHT/2):.3f} m")
            print(f"\nNote: Asymmetric position can affect vortex shedding.")
    
    interactive_pos = interactive(
        show_cylinder_position,
        cx=cx_slider,
        cy=cy_slider,
    )
    display(interactive_pos)

## 10. Spectral Analysis of Turbulent Wake

Analyze the frequency content of the turbulent wake to identify vortex shedding frequency.

In [ ]:
if HAS_MODEL:
    # Extract v-velocity time series at a probe point behind cylinder
    x_probe = 0.5  # Behind cylinder
    y_probe = 0.2  # Centerline
    
    x_idx = np.argmin(np.abs(x - x_probe))
    y_idx = np.argmin(np.abs(y - y_probe))
    
    # Get v-velocity time series (need to transpose)
    v_time_series = trajectory[:, x_idx, y_idx, 1]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Time series
    axes[0].plot(t, v_time_series, 'b-', linewidth=1)
    axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('v-velocity (m/s)')
    axes[0].set_title(f'Velocity Signal at Probe ({x_probe}, {y_probe})')
    axes[0].grid(True, alpha=0.3)
    
    # FFT
    dt = t[1] - t[0]
    freqs = np.fft.fftfreq(len(t), dt)
    fft_vals = np.abs(np.fft.fft(v_time_series - v_time_series.mean()))
    
    # Only positive frequencies
    pos_mask = freqs > 0
    axes[1].plot(freqs[pos_mask], fft_vals[pos_mask], 'r-', linewidth=1)
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_ylabel('Amplitude')
    axes[1].set_title('Frequency Spectrum')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(0, 5)
    
    plt.tight_layout()
    plt.show()
    
    # Estimate Strouhal number
    peak_freq = freqs[pos_mask][np.argmax(fft_vals[pos_mask])]
    D = 2 * CYLINDER_RADIUS
    U = model.U_mean
    St = peak_freq * D / U
    
    print(f"Dominant frequency: {peak_freq:.3f} Hz")
    print(f"Strouhal number: St = f*D/U = {St:.3f}")
    print(f"(Expected St ~ 0.2 for cylinder flow)")

## 11. Comparing LES vs No-LES

The Smagorinsky LES model adds turbulent viscosity to stabilize high-Re simulations.

In [ ]:
if HAS_MODEL:
    print("LES (Large Eddy Simulation) Model:")
    print("=" * 40)
    print(f"Enabled: {model.use_les}")
    print(f"Smagorinsky constant C_s: {model.C_s}")
    print(f"")
    print("Effect of LES:")
    print("- Adds turbulent viscosity: ν_t = (C_s·Δ)²|S|")
    print("- Stabilizes high Reynolds number simulations")
    print("- Captures energy cascade from large to small scales")
    print("- Dissipates energy at subgrid scales")
    print(f"")
    print(f"To disable LES, set use_les=False when creating the model.")
    print(f"Warning: High Re without LES may lead to numerical instability.")

## Summary

This notebook demonstrated:

1. **Turbulent cylinder flow** at high Reynolds numbers (Re ~ 500-1000)
2. **Smagorinsky LES** turbulence modeling
3. **Parameterized cylinder position** for varied flow configurations
4. **Interactive visualization** with ipywidgets
5. **Spectral analysis** of vortex shedding

### For Operator Learning

This dataset provides:
- **Input**: (inlet_velocity_scale, cylinder_x, cylinder_y)
- **Output**: Time trajectory (n_t, nx, ny, 3) with u, v, p

Learning tasks:
1. Predict flow evolution given cylinder position
2. Learn vortex shedding frequency vs Reynolds number
3. Transfer learning across different cylinder positions